<a href="https://colab.research.google.com/github/yudithvega-art/bhm-mrsa-indonesia/blob/main/Comparing_Zero_inflated%2BCensoring_v_Censoring_only.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model comparison: censoring-only vs zero-inflated + censored
All farms kept. Milk/Bioaerosol/Dust (water is 100% detected). Reports eta/tau/omega/sigma/alpha/delta + LOO.
Run: install -> upload MRSA_replicate_level_modelready.xlsx -> Run all.

In [1]:
!pip -q install "pymc>=5" arviz openpyxl pandas numpy

In [2]:
# ============================================================================
#  Compare LEFT-CENSORING-ONLY model vs ZERO-INFLATED + left-censored model.
#  Both keep ALL farms (incl. never-positive ones). Matrices with non-detections:
#  Milk (74), Bioaerosol (2), Dust (15). Water is 100% detected (skipped).
#  Reports every BHM parameter: eta (grand mean, log10), tau, omega, sigma,
#  and (ZI only) alpha (-> structural-zero prob pi) and delta.
#  COLAB: !pip -q install "pymc>=5" arviz openpyxl pandas numpy ; upload xlsx.
# ============================================================================
import numpy as np, pandas as pd, pymc as pm, arviz as az, pytensor.tensor as pt

df = pd.read_excel("MRSA_replicate_level_modelready.xlsx", sheet_name="replicate_data").dropna(subset=["matrix"])
ETA_PRIOR = {"Milk": (4.0, 1.2), "Bioaerosol": (2.0, 0.8), "Dust": (1.0, 0.8)}
UNIT = {"Milk": "CFU/mL", "Bioaerosol": "CFU/m3", "Dust": "CFU/cm2"}
DRAWS, TUNE, CHAINS, SEED = 1500, 1500, 2, 42

def _q(x):
    a, b, c = np.percentile(np.asarray(x), [2.5, 50, 97.5]); return float(b), float(a), float(c)
def fmt(q): return f"{q[0]:.3f} [{q[1]:.3f}, {q[2]:.3f}]"

def fit(m, zero_inflated, cores=1):
    d = df[df.matrix == m].copy().reset_index(drop=True)
    farms = sorted(d.farm.unique()); fidx = {f: i for i, f in enumerate(farms)}
    d["pt_id"] = d.farm.astype(str) + "||" + d.point.astype(str)
    points = sorted(d.pt_id.unique()); pidx = {p: i for i, p in enumerate(points)}
    J, K = len(farms), len(points)
    point_farm = np.array([fidx[p.split("||")[0]] for p in points]); d["k"] = d.pt_id.map(pidx)
    LOD = float(d.LOD.iloc[0]); logLOD = np.log10(LOD)
    k_obs = d.k.values.astype("int64"); is_det = (d.detection.values == 1).astype("float64")
    conc = d.conc.values.astype("float64")
    value = np.where(is_det > 0.5, np.log10(np.where(conc > 0, conc, 1.0)), logLOD)
    mu_eta, sd_eta = ETA_PRIOR[m]

    def obs_logp(value, mu_, pi_, sig_, isdet_, logLOD_):
        nll = pm.logp(pm.Normal.dist(mu_, sig_), value)
        det_ll = pt.log1p(-pi_) + nll                      # (1-pi)*Normal ; pi=0 -> Normal
        z_c = (logLOD_ - mu_) / sig_; Phi = 0.5 * pt.erfc(-z_c / pt.sqrt(2.0))
        cen_ll = pt.log(pi_ + (1.0 - pi_) * Phi + 1e-12)   # pi + (1-pi)*Phi ; pi=0 -> Phi
        return pt.where(isdet_ > 0.5, det_ll, cen_ll)

    with pm.Model() as model:
        eta = pm.Normal("eta", mu_eta, sd_eta)
        tau = pm.HalfNormal("tau", 0.6); omega = pm.HalfNormal("omega", 0.6); sigma = pm.HalfNormal("sigma", 1.0)
        z_theta = pm.Normal("z_theta", 0, 1, shape=J); theta = pm.Deterministic("theta", eta + tau * z_theta)
        z_mu = pm.Normal("z_mu", 0, 1, shape=K); mu = pm.Deterministic("mu", theta[point_farm] + omega * z_mu)
        if zero_inflated:
            delta = pm.HalfNormal("delta", 1.0); alpha = pm.Normal("alpha", 0, 1.5, shape=J)
            z_logit = pm.Normal("z_logit", 0, 1, shape=K)
            pi = pm.Deterministic("pi", pm.math.sigmoid(alpha[point_farm] + delta * z_logit))
        else:
            pi = pm.Deterministic("pi", pt.zeros(K))
        pm.CustomDist("obs", mu[k_obs], pi[k_obs], sigma, pt.as_tensor_variable(is_det), logLOD,
                      logp=obs_logp, observed=value)
        idata = pm.sample(draws=DRAWS, tune=TUNE, chains=CHAINS, cores=cores,
                          target_accept=0.97, random_seed=SEED, progressbar=False)
        pm.compute_log_likelihood(idata, model=model, progressbar=False)
    return idata

def summarise(idata, m, zi):
    p = idata.posterior; g = lambda v: p[v].values.reshape(-1)
    eta = g("eta"); gm = _q(10**eta)
    row = dict(matrix=m, model=("zero-inflated" if zi else "censoring-only"),
               grand_mean=f"{gm[0]:.4g} [{gm[1]:.3g}, {gm[2]:.4g}]",
               eta=fmt(_q(eta)), tau=fmt(_q(g("tau"))), omega=fmt(_q(g("omega"))), sigma=fmt(_q(g("sigma"))))
    if zi:
        pib = _q(p["pi"].values.mean(axis=-1).reshape(-1))     # mean structural-zero prob
        row["delta"] = fmt(_q(g("delta")))
        row["alpha_mean(logit)"] = fmt(_q(g("alpha")))         # pooled over farm draws
        row["struct_zero_pi"] = f"{pib[0]:.3f} [{pib[1]:.3f}, {pib[2]:.3f}]"
    else:
        row["delta"] = "\u2014 (no ZI)"; row["alpha_mean(logit)"] = "\u2014 (no ZI)"; row["struct_zero_pi"] = "0 (all non-det = censored)"
    dvg = int(idata.sample_stats["diverging"].values.sum())
    s = az.summary(idata, var_names=["eta","tau","omega","sigma"], round_to=3)
    row["max_rhat"] = float(s["r_hat"].max()); row["min_ess"] = float(s["ess_bulk"].min()); row["div"] = dvg
    return row, idata

if __name__ == "__main__":
    rows = {}; idatas = {}
    for m in ["Milk", "Bioaerosol", "Dust"]:
        for zi in [True, False]:
            print(f"fitting {m} | {'zero-inflated' if zi else 'censoring-only'} ...")
            idata = fit(m, zi); r, _ = summarise(idata, m, zi); rows[(m,zi)] = r; idatas[(m,zi)] = idata

    out = pd.DataFrame([rows[(m,zi)] for m in ["Milk","Bioaerosol","Dust"] for zi in [True,False]])
    cols = ["matrix","model","grand_mean","eta","tau","omega","sigma","delta","alpha_mean(logit)","struct_zero_pi","max_rhat","min_ess","div"]
    out = out[cols]; out.to_csv("model_comparison_ZI_vs_CO.csv", index=False)
    print("\n================  ALL PARAMETERS: zero-inflated vs censoring-only  ================")
    print(out.to_string(index=False))

    # LOO comparison per matrix + grand-mean difference
    print("\n================  MODEL COMPARISON (LOO-CV) & grand-mean shift  ================")
    for m in ["Milk","Bioaerosol","Dust"]:
        zi, co = idatas[(m,True)], idatas[(m,False)]
        cmp = az.compare({"zero_inflated": zi, "censoring_only": co})
        best = cmp.sort_values("rank").index[0]
        gm_zi = 10**np.median(zi.posterior["eta"].values); gm_co = 10**np.median(co.posterior["eta"].values)
        fold = gm_co/gm_zi
        print(f"\n{m}: LOO prefers -> {best.replace('_',' ')} | grand mean ZI={gm_zi:.4g} vs CO={gm_co:.4g} {UNIT[m]} "
              f"(CO/ZI = {fold:.2f}x)")
        print(cmp[["rank","elpd_diff","dse","weight"]].to_string())
    print("\nSaved: model_comparison_ZI_vs_CO.csv")


fitting Milk | zero-inflated ...
fitting Milk | censoring-only ...
fitting Bioaerosol | zero-inflated ...
fitting Bioaerosol | censoring-only ...
fitting Dust | zero-inflated ...
fitting Dust | censoring-only ...

================  ALL PARAMETERS: zero-inflated vs censoring-only  ================
    matrix          model          grand_mean                   eta                  tau                omega                sigma                delta       alpha_mean(logit)             struct_zero_pi  max_rhat  min_ess  div
      Milk  zero-inflated  80.18 [5.51, 4789]  1.904 [0.741, 3.680] 0.456 [0.020, 1.559] 1.676 [0.684, 2.327] 0.512 [0.353, 0.833] 1.801 [0.205, 3.501]   0.089 [-2.911, 3.106]       0.520 [0.341, 0.721]     1.007  434.994    0
      Milk censoring-only  5.16 [0.57, 33.37] 0.713 [-0.244, 1.523] 0.369 [0.018, 1.243] 1.845 [1.357, 2.443] 0.851 [0.614, 1.270]            — (no ZI)               — (no ZI) 0 (all non-det = censored)     1.006  894.602    0
Bioaerosol  zero-infl

/usr/local/lib/python3.13/dist-packages/arviz/stats/stats.py:797: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/arviz/stats/stats.py:797: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(



Milk: LOO prefers -> zero inflated | grand mean ZI=80.18 vs CO=5.16 CFU/mL (CO/ZI = 0.06x)
                rank  elpd_diff       dse    weight
zero_inflated      0   0.000000  0.000000  0.683023
censoring_only     1   2.841086  8.429937  0.316977


/usr/local/lib/python3.13/dist-packages/arviz/stats/stats.py:797: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/arviz/stats/stats.py:797: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(



Bioaerosol: LOO prefers -> censoring only | grand mean ZI=73.48 vs CO=73.25 CFU/m3 (CO/ZI = 1.00x)
                rank  elpd_diff       dse        weight
censoring_only     0   0.000000  0.000000  1.000000e+00
zero_inflated      1  12.079828  0.616355  3.730349e-14

Dust: LOO prefers -> zero inflated | grand mean ZI=8.349 vs CO=3.741 CFU/cm2 (CO/ZI = 0.45x)
                rank  elpd_diff       dse    weight
zero_inflated      0   0.000000  0.000000  0.628245
censoring_only     1   2.360614  5.001521  0.371755

Saved: model_comparison_ZI_vs_CO.csv
